In [ ]:
import sys
sys.path.append('../src')

from backgammon_env import *

In [ ]:
def encoder_case(nb_pions):
    """
    Transforme un nombre de pions (0 a 3) en 3 unites binaires,
    selon le principe de Tesauro simplifie pour l'Hypergammon.
    """
    unites = [0, 0, 0]
    for i in range(min(nb_pions, 3)):
        unites[i] = 1
    return unites


def encoder_etat(env):
    """
    Transforme l'etat complet de l'environnement en vecteur d'input
    pour le reseau de neurones. Retourne un array numpy de taille 149.
    """
    inputs = []
    
    for case in range(24):
        valeur = env.board[case]
        
        nb_blanc = int(valeur) if valeur > 0 else 0
        nb_noir = int(-valeur) if valeur < 0 else 0
        
        inputs.extend(encoder_case(nb_blanc))
        inputs.extend(encoder_case(nb_noir))
    
    inputs.append(env.bar_blanc / env.n_pions)
    inputs.append(env.bar_noir / env.n_pions)
    inputs.append(env.sortis_blanc / env.n_pions)
    inputs.append(env.sortis_noir / env.n_pions)
    
    inputs.append(1.0 if env.joueur_actuel == "blanc" else 0.0)
    
    return np.array(inputs)

In [ ]:
class ReseauValeur:
    """
    Reseau de neurones simple (MLP) : 149 inputs -> couche cachee -> 1 output.
    L'output est V(s), l'estimation de probabilite de victoire des BLANCS
    depuis l'etat s (entre 0 et 1).
    Implemente from scratch en numpy : forward pass + backpropagation.
    """
    
    def __init__(self, n_inputs=149, n_caches=40, alpha=0.1):
        self.alpha = alpha
        
        # Initialisation des poids : petites valeurs aleatoires autour de 0
        self.W1 = np.random.randn(n_inputs, n_caches) * 0.1   # poids entree -> cachee
        self.b1 = np.zeros(n_caches)                            # biais couche cachee
        self.W2 = np.random.randn(n_caches, 1) * 0.1            # poids cachee -> sortie
        self.b2 = np.zeros(1)                                    # biais sortie
    
    def sigmoid(self, x):
        """Fonction d'activation : ecrase n'importe quel nombre entre 0 et 1."""
        return 1.0 / (1.0 + np.exp(-x))
    
    def forward(self, x):
        """
        Passe avant : calcule V(s) a partir du vecteur d'etat x.
        On garde les valeurs intermediaires en memoire pour la backprop.
        """
        self.x = x
        self.z1 = x @ self.W1 + self.b1        # combinaison lineaire couche 1
        self.a1 = self.sigmoid(self.z1)         # activation couche cachee
        self.z2 = self.a1 @ self.W2 + self.b2  # combinaison lineaire couche 2
        self.a2 = self.sigmoid(self.z2)         # activation sortie = V(s)
        return self.a2[0]
    
    def backward(self, td_error):
        """
        Retropropagation : ajuste tous les poids dans la direction
        qui reduit la surprise (td_error), ponderee par alpha.
        """
        # Gradient de la sortie (derivee de la sigmoid : a*(1-a))
        delta2 = td_error * self.a2 * (1 - self.a2)
        
        # Gradients des poids de la couche de sortie
        dW2 = np.outer(self.a1, delta2)
        db2 = delta2
        
        # Propagation du gradient vers la couche cachee
        delta1 = (delta2 @ self.W2.T) * self.a1 * (1 - self.a1)
        dW1 = np.outer(self.x, delta1)
        db1 = delta1
        
        # Mise a jour des poids (montee de gradient sur V, ponderee par alpha)
        self.W1 += self.alpha * dW1
        self.b1 += self.alpha * db1.flatten()
        self.W2 += self.alpha * dW2
        self.b2 += self.alpha * db2.flatten()

In [ ]:
def choisir_meilleure_sequence(env, sequences, reseau):
    """
    Implemente le coeur de TD-Gammon : pour chaque sequence de coups possible,
    simuler l'etat resultant, l'evaluer avec le reseau, et choisir :
    - la sequence qui MAXIMISE V(s') si Blanc joue (V = proba victoire Blanc)
    - la sequence qui MINIMISE V(s') si Noir joue
    """
    meilleure_seq = None
    meilleure_valeur = None
    joueur = env.joueur_actuel
    
    for seq in sequences:
        # Simuler la sequence sur une copie de l'environnement
        env_simule = BackgammonEnv(n_pions=env.n_pions)
        env_simule.board = env.board.copy()
        env_simule.bar_blanc = env.bar_blanc
        env_simule.bar_noir = env.bar_noir
        env_simule.sortis_blanc = env.sortis_blanc
        env_simule.sortis_noir = env.sortis_noir
        env_simule.joueur_actuel = env.joueur_actuel
        
        for (depart, arrivee, de) in seq:
            env_simule.jouer_coup(depart, arrivee)
        
        valeur = reseau.forward(encoder_etat(env_simule))
        
        if meilleure_valeur is None:
            meilleure_seq, meilleure_valeur = seq, valeur
        elif joueur == "blanc" and valeur > meilleure_valeur:
            meilleure_seq, meilleure_valeur = seq, valeur
        elif joueur == "noir" and valeur < meilleure_valeur:
            meilleure_seq, meilleure_valeur = seq, valeur
    
    return meilleure_seq

In [ ]:
def evaluer_contre_random(reseau, n_parties=200):
    """
    L'agent entraine joue les BLANCS, un agent aleatoire joue les NOIRS.
    Retourne le winrate de l'agent entraine.
    """
    victoires = 0
    
    for _ in range(n_parties):
        env = BackgammonEnv()
        env.reset()
        
        for tour in range(500):
            des = lancer_des()
            sequences = env.coups_disponibles(des)
            
            if sequences:
                if env.joueur_actuel == "blanc":
                    seq = choisir_meilleure_sequence(env, sequences, reseau)
                else:
                    seq = sequences[np.random.randint(len(sequences))]
                for (depart, arrivee, de) in seq:
                    env.jouer_coup(depart, arrivee)
            
            gagnant = env.partie_terminee()
            if gagnant:
                if gagnant == "blanc":
                    victoires += 1
                break
            
            env.changer_joueur()
    
    return victoires / n_parties

In [ ]:
def entrainer_avec_suivi(n_parties=3000, alpha_debut=0.01, alpha_fin=0.001, eval_tous_les=500):
    reseau = ReseauValeur(n_inputs=149, n_caches=40, alpha=alpha_debut)
    historique_winrate = []
    meilleur_winrate = 0.0
    meilleurs_poids = None
    
    parties_par_bloc = eval_tous_les
    n_blocs = n_parties // parties_par_bloc
    
    for bloc in range(n_blocs):
        progression = bloc / n_blocs
        epsilon = 0.15 * (1 - progression) + 0.01
        # NOUVEAU : alpha decroit aussi au fil de l'entrainement
        reseau.alpha = alpha_debut + (alpha_fin - alpha_debut) * progression
        
        for partie in range(parties_par_bloc):
            env = BackgammonEnv()
            env.reset()
            etat_precedent = encoder_etat(env)
            V_precedent = reseau.forward(etat_precedent)
            
            for tour in range(500):
                des = lancer_des()
                sequences = env.coups_disponibles(des)
                if sequences:
                    if np.random.rand() < epsilon:
                        seq = sequences[np.random.randint(len(sequences))]
                    else:
                        seq = choisir_meilleure_sequence(env, sequences, reseau)
                    for (depart, arrivee, de) in seq:
                        env.jouer_coup(depart, arrivee)
                
                gagnant = env.partie_terminee()
                if gagnant:
                    reward = 1.0 if gagnant == "blanc" else 0.0
                    td_error = reward - V_precedent
                    reseau.forward(etat_precedent)
                    reseau.backward(td_error)
                    break
                
                env.changer_joueur()
                etat_nouveau = encoder_etat(env)
                V_nouveau = reseau.forward(etat_nouveau)
                td_error = V_nouveau - V_precedent
                reseau.forward(etat_precedent)
                reseau.backward(td_error)
                etat_precedent = etat_nouveau
                V_precedent = V_nouveau
        
        winrate = evaluer_contre_random(reseau, n_parties=100)
        historique_winrate.append(winrate)
        
        # NOUVEAU : on garde une copie des poids du meilleur reseau vu jusqu'ici
        if winrate > meilleur_winrate:
            meilleur_winrate = winrate
            meilleurs_poids = (reseau.W1.copy(), reseau.b1.copy(), reseau.W2.copy(), reseau.b2.copy())
        
        print(f"Après {(bloc+1)*parties_par_bloc} parties — alpha : {reseau.alpha:.4f} — winrate : {winrate:.1%}")
    
    # A la fin : on restaure le MEILLEUR reseau, pas le dernier
    if meilleurs_poids is not None:
        reseau.W1, reseau.b1, reseau.W2, reseau.b2 = meilleurs_poids
        print(f"\nMeilleur réseau restauré (winrate : {meilleur_winrate:.1%})")
    
    return reseau, historique_winrate

In [ ]:
n_points = len(winrates)
x_axis = [500 * (i + 1) for i in range(n_points)]

plt.figure(figsize=(8, 4))
plt.plot(x_axis, winrates, marker='o')
plt.axhline(y=0.5, color='gray', linestyle='--', label='Hasard (50%)')
plt.axhline(y=0.7, color='green', linestyle='--', label='Jalon roadmap (70%)')
plt.xlabel("Parties d'entraînement")
plt.ylabel("Winrate vs random")
plt.title("Progression du winrate")
plt.legend()
plt.show()